# 0. Setup: environment, work directory, and what each notebook needs

This repository accompanies *A compact core motif code is differentially interpreted across human cell types*. CNN-CR learns transcription-factor motifs de novo while predicting gene expression from a gene's promoter and enhancer windows; across 116 EpiMap biosamples the per-biosample motif sets collapse into 43 core promoter motifs.

Each notebook is one analysis step, runs the archived scripts in `code/` from the right folder and shows what they produce. **K562 (biosample BSS00762) is the worked example** wherever a step runs per biosample.

| notebook | what it does | figure |
|---|---|---|
| 01_train_cnncr | window FASTA -> one-hot tensor -> train and test the cell-type-specific CNN-CR model of K562 | Fig. 1c,d |
| 02_motif_extraction_and_analysis | extract the motifs of the trained model; merge the ten K562 models' motifs into the non-redundant set; motif yield and catalogue concordance of 116 biosamples | Fig. 1e,f |
| 03_linear_model | motif count matrices -> ridge model -> held-out predictions (K562) | Fig. 1h |
| 04_core_motifs | the 43 core motifs start to end from the per-biosample motifs of the 116 biosamples; JASPAR readers; control sets | Fig. 2a-c |
| 05_core43_cnncr | the 43-motif CNN-CR trained and tested start to end on K562 | Fig. 2e |
| 06_fig3_conservation | phyloP at core-motif sites vs ten control sets; reader-protein constraint | Fig. 3c-h |
| 07_eqtl_variants | fine-mapped causal eQTLs at core versus control motif sites, 16 tissues | Fig. 4a,b |
| 08_chipseq_occupancy | ENCODE peaks -> fraction of each TF's peaks with a core vs control site | Fig. 4d |
| 09_hek293_heldout | HEK293 held-out: linear, random forest and the 43-motif CNN-CR | Fig. 4e |
| 10_reader_sites_chipseq | which candidate reader binds each shared core-motif site in five cell lines | Fig. 5b,c |
| 11_reader_switching_perturbseq | reader dominance and switching from the quantile-normalised Perturb-seq screens | Fig. 5d-g |

**Setup, once:** build the environment (`conda env create -f environment.yml`), then `python bin/setup_work_dir.py --enhancer-root /abs/path/work` (creates the work directory, links the small shipped inputs, rewrites the scripts' paths). Unpack the Zenodo archives you need into the work directory (`tar -xzf <archive> -C $ENHANCER_ROOT`; three archives come as `.partNN` pieces to join first with `cat <archive>.tar.gz.part* > <archive>.tar.gz`, see `external_data/ZENODO_MANIFEST.md`) and, for notebooks 08/10/11, download the ENCODE peaks with `bin/download_encode_peaks.py`. The cells below check all of it.

In [ ]:
# ---- Shared setup (identical in every notebook) --------------------------------------------------------
import os, sys, json, subprocess, time, glob, shutil
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))          # notebooks/ lives one level below the repository root
CODE = os.path.join(REPO, "code")
roots = json.load(open(os.path.join(REPO, ".roots.json")))          # written by bin/rewrite_paths.py (see PATHS.md)
ENHANCER_ROOT, SEQ_ROOT = roots["enhancer_root"], roots["seq_root"]
def run(cmd, cwd=None, env=None):
    """Run one repository script from the right folder and stream its output here. Stops the notebook if it fails."""
    print("$", cmd, "   (in", cwd or os.getcwd(), ")"); t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout: print(line, end="")
    p.wait(); print(f"[exit {p.returncode}, {time.time()-t:.0f} s]")
    assert p.returncode == 0, "this step failed - read the output above"
def n_motifs(meme):
    """Number of MOTIF entries in a MEME file."""
    return sum(1 for l in open(meme) if l.startswith("MOTIF")) if os.path.exists(meme) else None
def show_logos(meme, n=5, title=""):
    """Sequence logos of the first n motifs of a MEME file (logomaker)."""
    import numpy as np, pandas as pd, matplotlib.pyplot as plt, logomaker
    blocks = open(meme).read().split("MOTIF ")[1:n+1]; fig, axes = plt.subplots(1, len(blocks), figsize=(2.2*len(blocks), 1.4))
    for ax, b in zip(np.atleast_1d(axes), blocks):
        rows = [l.split() for l in b.split("\n")[2:] if l.strip() and not l.startswith("letter") and not l.startswith("URL")]
        pwm = pd.DataFrame([[float(x) for x in r] for r in rows if len(r) == 4], columns=list("ACGT"))
        logomaker.Logo(logomaker.transform_matrix(pwm, from_type="probability", to_type="information"), ax=ax); ax.set_title(b.split("\n")[0], fontsize=8); ax.set_yticks([]); ax.set_xticks([])
    fig.suptitle(title, fontsize=9); plt.tight_layout(); plt.show()
print("repository:", REPO); print("ENHANCER_ROOT:", ENHANCER_ROOT); print("SEQ_ROOT:", SEQ_ROOT)


## Environment check
Every import the notebooks need, TOMTOM and bedtools, and whether a GPU is visible (notebooks 01, 05 and 09 train networks; on a CPU they take an hour or more).

In [ ]:
import importlib, shutil
for m in ["numpy","pandas","scipy","sklearn","matplotlib","networkx","anndata","pyarrow","pyBigWig","logomaker","torch","nbformat"]:
    try: importlib.import_module(m); print(f"  ok  {m}")
    except Exception as e: print(f"  MISSING {m}: {e}")
for tool in ["tomtom","bedtools"]: print(f"  {'ok ' if shutil.which(tool) else 'MISSING'} {tool}")
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Work directory and data
`PATHS.md` explains the layout: the small inputs ship in the repository and are linked into the work directory by `bin/setup_work_dir.py`; the large inputs are the Zenodo archives (`external_data/ZENODO_MANIFEST.md`) and the ENCODE peak download. Each line below names the notebook that needs the item.

In [ ]:
checks = [
 ("all",      "work directory links code/ as multi_agent",           f"{ENHANCER_ROOT}/multi_agent/posthoc_analysis"),
 ("all",      "biosample list (repository)",                          f"{ENHANCER_ROOT}/Epimap_BSSID_nondup.txt"),
 ("01,02,05", "K562 training input (repository)",                     f"{REPO}/data/k562_demo/BSS00762.windows.fa.gz"),
 ("01,02,05", "other biosamples: Zenodo fig1_training_inputs_all116", f"{ENHANCER_ROOT}/fig1_training_inputs"),
 ("02",       "Zenodo fig1_motif_sets_all116",                        f"{ENHANCER_ROOT}/fig1_motif_sets/biosamples.tsv"),
 ("04",       "Zenodo fig2_core_motif_derivation",                    f"{ENHANCER_ROOT}/cell_type_specific_CNN/BSS00762/BSS00762_it_0_256MOTIF_pytorch_0_pro_motifs_all_256.meme"),
 ("06",       "Zenodo fig3_conservation_inputs",                      f"{CODE}/conservation_sequence/hg38.phyloP100way.promoter_windows.bw"),
 ("07",       "Zenodo fig4_eqtl_inputs",                              f"{ENHANCER_ROOT}/enformer_comparison/eqtl-enformer"),
 ("08,11",    "Zenodo fig4_fig5_motif_sites_on_peaks",                f"{CODE}/chipseq_enrichment/results/fimo_poolscan/K562/fimo_peaks/fimo.tsv"),
 ("10,11",    "Zenodo fig5_reader_switch_inputs",                     f"{ENHANCER_ROOT}/perturbseq_qn/perturbseq_qn_K562.npz"),
 ("08,10,11", "ENCODE peaks (bin/download_encode_peaks.py)",          f"{ENHANCER_ROOT}/encode4_tf_chipseq/K562"),
]
for nbs, what, path in checks: print(f"  {'ok ' if os.path.exists(path) else '-- '} [{nbs:8s}] {what}: {path}")
print("\nBiosamples:", len(open(f"{ENHANCER_ROOT}/Epimap_BSSID_nondup.txt").read().split()))

## The Zenodo archives

In [ ]:
print(open(os.path.join(REPO, "external_data", "ZENODO_MANIFEST.md")).read())